# Olist: delivery time

## Goal
How many days pass between purchase and delivery to the customer? Define the scope, look at missing values, then average by month. The SQL version is in `sql/04_delivery_days.sql`.

## Setup
Load orders and parse the two timestamps (failures become NaT instead of being dropped silently).

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Project folder = the nearest parent that contains data/raw
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
DATA_DIR = ROOT / "data" / "raw"

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
orders["purchase_ts"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
orders["delivered_ts"] = pd.to_datetime(orders["order_delivered_customer_date"], errors="coerce")
print("orders:", orders.shape)

orders: (99441, 10)


## Definitions
- **Orders in scope**: `order_status == "delivered"` with both `delivered_ts` and `purchase_ts` present. Status alone would include 8 orders without a delivery timestamp; the timestamp alone would include 6 canceled orders that have one. Both conditions are used.
- **Delivery days**: `delivered_ts − purchase_ts`, in days with decimals (`total_seconds() / 86400`). No rounding, so "3 days 20 hours" is not counted as 3 days.
- **Missing values**: not imputed. Affected orders are excluded and counted.
- **Checks**: 96,470 orders in scope (96,478 delivered − 8 without a delivery timestamp); no negative `delivery_days`; the months add up to 96,470.

## Missing values and distribution

In [2]:
# 1. Count missing values first
is_delivered = orders["order_status"] == "delivered"
print("Orders with status delivered:", is_delivered.sum())
print("  of which without a delivery timestamp:", (is_delivered & orders["delivered_ts"].isna()).sum())
print("Not delivered but with a delivery timestamp:", (~is_delivered & orders["delivered_ts"].notna()).sum())
print("Missing purchase timestamp:", orders["purchase_ts"].isna().sum())

# 2. Scope: delivered, with both timestamps
delivered = orders[
    is_delivered & orders["delivered_ts"].notna() & orders["purchase_ts"].notna()
].copy()
print("Orders in scope:", len(delivered))

# 3. Delivery time in days, with decimals
delivered["delivery_days"] = (
    (delivered["delivered_ts"] - delivered["purchase_ts"]).dt.total_seconds() / 86400
)

# 4. Distribution: any negative or extreme values?
print("Negative values:", (delivered["delivery_days"] < 0).sum())
display(delivered["delivery_days"].describe().round(2).to_frame("delivery_days"))

Orders with status delivered: 96478
  of which without a delivery timestamp: 8
Not delivered but with a delivery timestamp: 6
Missing purchase timestamp: 0
Orders in scope: 96470
Negative values: 0


,delivery_days
count,96470.00
mean,12.56
std,9.55
min,0.53
25%,6.77
50%,10.22
75%,15.72
max,209.63


## Average delivery time by month

In [3]:
# Average by purchase month; the order count is shown because months with few orders support no conclusion
delivered["purchase_month"] = delivered["purchase_ts"].dt.to_period("M")

monthly_delivery = (
    delivered.groupby("purchase_month")["delivery_days"]
    .agg(n_orders="size", avg_days="mean", median_days="median")
    .round(2)
)
print(monthly_delivery.to_string())

# Check: the months add up to the orders in scope
print("Sum of months:", monthly_delivery["n_orders"].sum(), "| orders in scope:", len(delivered),
      "| equal:", monthly_delivery["n_orders"].sum() == len(delivered))

                n_orders  avg_days  median_days
purchase_month                                 
2016-09                1     54.81        54.81
2016-10              265     19.60        17.94
2016-12                1      4.69         4.69
2017-01              750     12.65        10.73
2017-02             1653     13.17        10.97
2017-03             2546     12.95        10.13
2017-04             2303     14.92        12.75
2017-05             3545     11.32         9.79
2017-06             3135     12.01        10.80
2017-07             3872     11.59        10.16
2017-08             4193     11.15         9.77
2017-09             4150     11.85        10.31
2017-10             4478     11.86        10.26
2017-11             7288     15.16        12.70
2017-12             5513     15.39        13.14
2018-01             7069     14.08        11.83
2018-02             6555     16.95        14.26
2018-03             7003     16.30        13.37
2018-04             6798     11.50      

## Findings
- Across the 96,470 delivered orders the mean delivery time is 12.56 days and the median 10.22 days; the longest is 209.63 days, so the mean is pulled up by a long tail.
- By purchase month, within the observation period the average was highest for orders placed in 2018-02 (16.95 days) and 2018-03 (16.30 days), and lowest in 2018-08 (7.73 days). Orders placed in the peak months 2017-11 and 2017-12 also took longer than average (15.16 and 15.39 days).
- An order is counted in the month it was placed. Orders placed near the end of the data that were still undelivered at the cut-off are missing from the last months, which can make those months look faster than they were.